In [ ]:
import cv2
import numpy as np
import os
import glob
from ultralytics import YOLO
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import joblib
from tqdm import tqdm

# --- 1. CẤU HÌNH ---
DATASET_PATH = 'dataset/' # Đường dẫn tới thư mục dataset (chứa train/ và val/)
YOLO_MODEL_PATH = 'yolo11n-pose.pt' # Đường dẫn tới model YOLO pose
CLASSIFIER_SAVE_PATH = 'posture_classifier.joblib' # Nơi lưu model phân loại SVM
SCALER_SAVE_PATH = 'posture_scaler.joblib' # Nơi lưu bộ chuẩn hóa StandardScaler

# Kích thước ảnh đầu vào (phù hợp OV2640)
INPUT_WIDTH = 320
INPUT_HEIGHT = 240
TARGET_SIZE = (INPUT_WIDTH, INPUT_HEIGHT)

# Các keypoints quan trọng (ví dụ) - Đảm bảo index 0 là mũi nếu chuẩn hóa theo mũi
IMPORTANT_KEYPOINTS_INDICES = [0, 5, 6, 11, 12] # Mũi, Vai trái/phải, Hông trái/phải

# --- 2. HÀM TRÍCH XUẤT ĐẶC TRƯNG TƯ THẾ ---
def extract_pose_features(image_path, yolo_model):
    """Trích xuất và chuẩn hóa tọa độ keypoints quan trọng."""
    try:
        img = cv2.imread(image_path)
        if img is None: return None
        img_resized = cv2.resize(img, TARGET_SIZE)
        results = yolo_model(img_resized, verbose=False)

        if results and results[0].keypoints and results[0].keypoints.shape[1] > 0:
            keypoints = results[0].keypoints.xy[0].cpu().numpy()
            features = []
            for idx in IMPORTANT_KEYPOINTS_INDICES:
                if idx < len(keypoints) and keypoints[idx][0] > 1 and keypoints[idx][1] > 1:
                    features.extend(keypoints[idx])
                else:
                    features.extend([0, 0]) # Dùng 0,0 nếu thiếu

            # Chuẩn hóa theo mũi (nếu mũi là index 0)
            if len(features) > 1 and IMPORTANT_KEYPOINTS_INDICES[0] == 0 and features[0] != 0 and features[1] != 0:
                 nose_x, nose_y = features[0], features[1]
                 normalized_features = []
                 for i in range(0, len(features), 2):
                      normalized_features.append(features[i] - nose_x if features[i] != 0 else 0)
                      normalized_features.append(features[i+1] - nose_y if features[i+1] != 0 else 0)
                 return np.array(normalized_features[2:]) # Bỏ mũi (luôn là 0,0 sau chuẩn hóa)
            elif len(features) > 0:
                 return np.array(features) # Trả về không chuẩn hóa nếu không có mũi
            else: return None
        else: return None
    except Exception as e:
        # print(f"Error processing {image_path}: {e}") # Bỏ comment để debug nếu cần
        return None

# --- 3. TẢI DỮ LIỆU VÀ TRÍCH XUẤT ĐẶC TRƯNG ---
print("Đang tải mô hình YOLO...")
model = YOLO(YOLO_MODEL_PATH)
print("-> Mô hình YOLO đã tải.")

def load_features_from_split(split_name, yolo_model):
    """Tải ảnh, trích xuất đặc trưng và gán nhãn cho một tập (train/val)."""
    features_list = []
    labels_list = []
    processed_count = 0
    skipped_count = 0
    split_path = os.path.join(DATASET_PATH, split_name)
    if not os.path.isdir(split_path):
        print(f"Không tìm thấy thư mục split '{split_name}'.")
        return None, None, 0, 0

    print(f"\nĐang xử lý split '{split_name}'...")
    for label_name in ['correct', 'incorrect']:
        label_dir = os.path.join(split_path, label_name)
        if not os.path.isdir(label_dir): continue

        label = 0 if label_name == 'correct' else 1
        image_files = glob.glob(os.path.join(label_dir, '*.jpg')) + \
                      glob.glob(os.path.join(label_dir, '*.png')) + \
                      glob.glob(os.path.join(label_dir, '*.jpeg'))
        print(f"Tìm thấy {len(image_files)} ảnh trong {label_dir}")

        split_processed = 0
        split_skipped = 0
        for img_path in tqdm(image_files, desc=f"Trích xuất đặc trưng {label_name} ({split_name})"):
            features = extract_pose_features(img_path, yolo_model)
            if features is not None:
                features_list.append(features)
                labels_list.append(label)
                split_processed += 1
            else:
                split_skipped += 1

        processed_count += split_processed
        skipped_count += split_skipped
        print(f"-> Trích xuất: {split_processed} ảnh, Bỏ qua: {split_skipped} ảnh.")

    return features_list, labels_list, processed_count, skipped_count

# Tải features cho tập train và validation
train_features_list, train_labels_list, train_proc, train_skip = load_features_from_split('train', model)
val_features_list, val_labels_list, val_proc, val_skip = load_features_from_split('val', model)

if not train_features_list:
    print("\n❌ Lỗi: Không trích xuất được đặc trưng nào từ tập 'train'. Dừng lại.")
    exit()

X_train = np.array(train_features_list)
y_train = np.array(train_labels_list)

X_val, y_val = None, None
if val_features_list:
    X_val = np.array(val_features_list)
    y_val = np.array(val_labels_list)
else:
    print("\n⚠️ Cảnh báo: Không tìm thấy hoặc xử lý được tập 'val'. Sẽ bỏ qua bước đánh giá.")

# --- KIỂM TRA ĐỘ DÀI VECTOR ĐẶC TRƯNG ---
# Chiều dài mong đợi = (số keypoints quan trọng - 1 nếu bỏ mũi) * 2 (x,y)
expected_length = (len(IMPORTANT_KEYPOINTS_INDICES) - (1 if IMPORTANT_KEYPOINTS_INDICES[0] == 0 else 0) ) * 2
valid_train_indices = [i for i, feature in enumerate(X_train) if len(feature) == expected_length]

if len(valid_train_indices) != len(X_train):
    print(f"\n⚠️ Cảnh báo (Train): Độ dài vector không đồng nhất. Sử dụng {len(valid_train_indices)} mẫu hợp lệ.")
    X_train = X_train[valid_train_indices]
    y_train = y_train[valid_train_indices]

if X_val is not None:
    valid_val_indices = [i for i, feature in enumerate(X_val) if len(feature) == expected_length]
    if len(valid_val_indices) != len(X_val):
        print(f"\n⚠️ Cảnh báo (Val): Độ dài vector không đồng nhất. Sử dụng {len(valid_val_indices)} mẫu hợp lệ.")
        X_val = X_val[valid_val_indices]
        y_val = y_val[valid_val_indices]

if len(X_train) == 0:
    print("\n❌ Lỗi: Không còn đặc trưng hợp lệ nào trong tập train.")
    exit()

print(f"\nTổng đặc trưng train hợp lệ: {len(X_train)}")
print(f"Chiều dài vector đặc trưng: {X_train.shape[1]}")
print(f"Phân bố nhãn Train: Đúng (0): {np.sum(y_train == 0)}, Sai (1): {np.sum(y_train == 1)}")
if X_val is not None:
     print(f"Tổng đặc trưng val hợp lệ: {len(X_val)}")
     print(f"Phân bố nhãn Val: Đúng (0): {np.sum(y_val == 0)}, Sai (1): {np.sum(y_val == 1)}")

# --- 4. CHUẨN HÓA DỮ LIỆU (Fit trên train, transform cả train và val) ---
print("\nĐang chuẩn hóa dữ liệu...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # Học (fit) và biến đổi (transform) tập train
joblib.dump(scaler, SCALER_SAVE_PATH)
print(f"-> Scaler đã được học trên train và lưu vào {SCALER_SAVE_PATH}")

X_val_scaled = None
if X_val is not None and len(X_val) > 0 : # Kiểm tra X_val có dữ liệu không
    X_val_scaled = scaler.transform(X_val) # Chỉ biến đổi (transform) tập val bằng scaler đã học
    print("-> Dữ liệu validation đã được chuẩn hóa.")
else:
    print("-> Bỏ qua chuẩn hóa validation do không có dữ liệu.")

# --- 5. HUẤN LUYỆN BỘ PHÂN LOẠI (SVM - Chỉ trên train) ---
print("\nĐang huấn luyện SVM trên tập train...")
# Bạn có thể thử nghiệm các tham số kernel, C, gamma khác nhau
classifier = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
classifier.fit(X_train_scaled, y_train)
print("-> Huấn luyện hoàn tất.")

# --- 6. ĐÁNH GIÁ MODEL (Trên validation) ---
if X_val_scaled is not None and y_val is not None and len(X_val_scaled) > 0:
    print("\nĐang đánh giá mô hình trên tập validation...")
    y_pred_val = classifier.predict(X_val_scaled)
    accuracy_val = accuracy_score(y_val, y_pred_val)
    report_val = classification_report(y_val, y_pred_val, target_names=['correct', 'incorrect'], zero_division=0) # Thêm zero_division

    print(f"Độ chính xác (Validation): {accuracy_val * 100:.2f}%")
    print("Báo cáo phân loại (Validation):\n", report_val)
else:
    print("\nBỏ qua đánh giá do không có tập validation hợp lệ.")

# --- 7. LƯU MODEL PHÂN LOẠI ---
joblib.dump(classifier, CLASSIFIER_SAVE_PATH)
print(f"\n-> Mô hình phân loại đã huấn luyện được lưu vào {CLASSIFIER_SAVE_PATH}")

print("\n--- QUÁ TRÌNH HUẤN LUYỆN HOÀN TẤT ---")